# This notebook covers how to do inference with a State Transition model pretrained on the Tahoe-100M dataset.

Due to storage and ram limits, it is highly recommended to run this with colab pro, or download the notebook to run it locally.
Note to Jeannie: running with the `relearn-env` python environment on chimera.

# Installation

In [1]:
! pip install -q anndata

import os, sys, pickle
import anndata as ad

os.environ['MPLBACKEND'] = 'agg'

# Download a pre-trained ST-Tahoe checkpoint

In [2]:
# 1) Install dependency
%pip install -q --upgrade huggingface_hub

# 2) Download only the specific subfolder
from huggingface_hub import snapshot_download

local_dir = snapshot_download(
    repo_id="arcinstitute/ST-HVG-Tahoe",
    allow_patterns="fewshot/state_generalization_X_hvg/*",
    local_dir="ST-HVG-Tahoe",
    local_dir_use_symlinks=False,
)

print(f"Downloaded to: {local_dir}")

Note: you may need to restart the kernel to use updated packages.


/home/jeannie/miniconda/envs/relearn-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jeannie/miniconda/envs/relearn-env/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching 48 files: 100%|██████████| 48/48 [00:00<00:00, 1618.65it/s]

Downloaded to: /home/jeannie/relearn/notebooks/jeannie/ST-HVG-Tahoe


# Fetch a small file from HuggingFace

In [3]:
# Download a cell line that was heldout from training the model
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="arcinstitute/State-Tahoe-Filtered",
    repo_type="dataset",
    filename="c37.h5ad",
    local_dir=".",  # downloads to current directory
    local_dir_use_symlinks=False
)

print(f"Downloaded to: {file_path}")

/home/jeannie/miniconda/envs/relearn-env/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Downloaded to: /home/jeannie/relearn/notebooks/jeannie/c37.h5ad


# Now we are ready to run inference on the file:

In [5]:
import time
start = time.time()

! uvx -q --from git+https://github.com/ArcInstitute/state@main state tx infer \
    --model-dir ST-HVG-Tahoe/fewshot/state_generalization_X_hvg/ \
    --checkpoint ST-HVG-Tahoe/fewshot/state_generalization_X_hvg/checkpoints/best.ckpt \
    --pert-col drugname_drugconc \
    --batch-col plate \
    --control-pert "[('DMSO_TF', 0.0, 'uM')]" \
    --embed-key "X_hvg" \
    --adata c37.h5ad \
    --output c37_simulated.h5ad

elapsed = time.time() - start
print(f"Inference time: {elapsed:.1f}s")

==> STATE: tx infer (virtual experiment)
Loaded config: ST-HVG-Tahoe/fewshot/state_generalization_X_hvg/config.yaml
Control perturbation: [('DMSO_TF', 0.0, 'uM')]
Grouping by cell type column: cell_name
Loaded batch one-hot map from: ST-HVG-Tahoe/fewshot/state_generalization_X_hvg/batch_onehot_map.pkl
Loaded cell type one-hot map from: ST-HVG-Tahoe/fewshot/state_generalization_X_hvg/cell_type_onehot_map.pkl
Traceback (most recent call last):
  File "/home/jeannie/.cache/uv/archive-v0/r-ocPL4rpdhZaL-D/bin/state", line 12, in <module>
    sys.exit(main())
             ^^^^^^
  File "/home/jeannie/.cache/uv/archive-v0/r-ocPL4rpdhZaL-D/lib/python3.11/site-packages/state/__main__.py", line 125, in main
    run_tx_infer(args)
  File "/home/jeannie/.cache/uv/archive-v0/r-ocPL4rpdhZaL-D/lib/python3.11/site-packages/state/_cli/_tx/_infer.py", line 440, in run_tx_infer
    model = StateTransitionPerturbationModel.load_from_checkpoint(checkpoint_path)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

# Next, copy the HVG predictions into a .X for a new AnnData for Cell-Eval

In [ ]:
import anndata as ad

filenames = ["c37.h5ad", "c37_simulated.h5ad"]

for fname in filenames:
    adata = ad.read_h5ad(fname)
    adata_hvg = ad.AnnData(
        X=adata.obsm["X_hvg"],
        obs=adata.obs.copy(),
    )
    adata_hvg.write_h5ad(fname.replace(".h5ad", "_hvg.h5ad"))

# Run Cell-Eval to compare observed data vs State simulated data

In [ ]:
# Run cell-eval to compare the simulated anndata vs the observed anndata.
start = time.time()
! uvx -q --from git+https://github.com/ArcInstitute/cell-eval@main cell-eval run \
    -ap c37_simulated.h5ad \
    -ar c37.h5ad \
    -o c37_eval \
    --control-pert "[('DMSO_TF', 0.0, 'uM')]" \
    --pert-col drugname_drugconc \
    --profile vcc \
    --celltype-col cell_name \
    --skip-metrics clustering_agreement,pearson_edistance \
    --batch-size 1024 \
    --num-threads 64

elapsed = time.time() - start
print(f"Inference time: {elapsed:.1f}s")

INFO:cell_eval._evaluator:Input is found to be log-normalized already - skipping transformation.
INFO:cell_eval._evaluator:Input is found to be log-normalized already - skipping transformation.
INFO:cell_eval._evaluator:Computing DE for real data
INFO:pdex._single_cell:Precomputing masks for each target gene
Identifying target masks: 100% 1137/1137 [00:01<00:00, 938.88it/s]
INFO:pdex._single_cell:Precomputing variable indices for each feature
Identifying variable indices: 100% 2000/2000 [00:00<00:00, 3133585.36it/s]
INFO:pdex._single_cell:Creating shared memory memory matrix for parallel computing
/usr/lib/python3.12/multiprocessing/resource_tracker.py:279: UserWarning: resource_tracker: There appear to be 1 leaked shared_memory objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '


In [ ]:
import pandas as pd

results = pd.read_csv('/content/NCI-H596_agg_NCI-H596_results.csv')
results[results.statistic == 'mean']

,statistic,overlap_at_N,mae,discrimination_score_l1
2,mean,0.000178,0.150929,0.501992
